# 操作客户端

In [6]:
from pymilvus import MilvusClient

client = MilvusClient("http://localhost:19530")

# 查看数据库

In [10]:
existed_databases = client.list_databases()

for db in existed_databases:
    print(db)

default
rag_demo


# 创建数据库

In [8]:
db_name = "rag_demo"

if db_name not in existed_databases:
    client.create_database(db_name=db_name)

# 删除数据库

In [5]:

client.drop_database(db_name = db_name)

# 切换数据库

In [9]:
client.use_database(db_name=db_name)

# 查看数据库下的collections

In [15]:
collections = client.list_collections()

for coll in collections:
    print(coll)

# 创建collection

In [12]:
collection_name = "docs"

client.create_collection(
    collection_name=collection_name,
    dimension=1024,
    metric_type="COSINE"
)

# 删除collection

In [14]:
client.drop_collection(collection_name=collection_name)

In [22]:
import os
from dotenv import load_dotenv
from langchain.embeddings import init_embeddings


load_dotenv(override=True)


embed_model =  init_embeddings(
    model="openai:text-embedding-v4",
    api_key=os.getenv("TONGYI_API_KEY"),
    base_url=os.getenv("TONGYI_BASE_URL"),
    check_embedding_ctx_length=False,
)

In [17]:
collection_name = "docs"

client.create_collection(
    collection_name=collection_name,
    dimension=1024,
    metric_type="COSINE"
)

In [18]:
from rich import print as rprint

metadata = client.describe_collection(collection_name=collection_name)

rprint(metadata)

{
    'collection_name': 'docs',
    'auto_id': False,
    'num_shards': 1,
    'description': '',
    'fields': [
        {
            'field_id': 100,
            'name': 'id',
            'description': '',
            'type': <DataType.INT64: 5>,
            'params': {},
            'is_primary': True
        },
        {
            'field_id': 101,
            'name': 'vector',
            'description': '',
            'type': <DataType.FLOAT_VECTOR: 101>,
            'params': {'dim': 1024}
        }
    ],
    'functions': [],
    'aliases': [],
    'collection_id': 467695520007328343,
    'consistency_level': 2,
    'properties': {'timezone': 'UTC'},
    'num_partitions': 1,
    'enable_dynamic_field': True,
    'enable_namespace': False,
    'created_timestamp': 467695699779649554,
    'update_timestamp': 467695699779649554
}

In [19]:
# 准备测试数据
texts = [
    "LangChain 是一个用于构建 LLM 应用的开发框架。",
    "Milvus 是一个适合 AI 应用的向量数据库。",
    "RAG 的核心是先检索相关知识，再让大模型生成答案。",
    "Docker Desktop 可以方便地在本地运行 Milvus Standalone。"
]

In [23]:
vectors = embed_model.embed_documents(texts)

In [24]:
print(len(vectors))

print(len(vectors[0]))

print(vectors[0][:5])

4
1024
[-0.06995262950658798, -0.03449718654155731, -0.038969043642282486, 0.0352691151201725, -0.02789587341248989]


In [25]:
data = [
    {
        "id" : i,
        "vector" : vectors[i],
        "text" : texts[i],
        "source" : "demo"
    } for i in range(len(texts))
]

In [26]:
insert_res = client.upsert(
    collection_name=collection_name,
    data=data,
)

print("insert result : ",insert_res)

insert result :  {'upsert_count': 4, 'ids': [0, 1, 2, 3]}


In [27]:
client.flush(collection_name=collection_name)

In [28]:
stats = client.get_collection_stats(collection_name=collection_name)

print("stats : ",stats)

stats :  {'row_count': 4}


In [29]:
iterator = client.query_iterator(
    collection_name=collection_name,
    filter="",
    output_fields=["*"]
)

i = 0
while True:

    rows = iterator.next()

    if not rows:
        break

    for row in rows:
        print(f"第{i + 1}条数据：")
        # print(row)

        print(f"id : {row["id"]},vector = {row["vector"][:5]},text = {row["text"]},source = {row["source"]}")

        i += 1

iterator.close()

第1条数据：
id : 0,vector = [-0.06995262950658798, -0.03449718654155731, -0.038969043642282486, 0.0352691151201725, -0.02789587341248989],text = LangChain 是一个用于构建 LLM 应用的开发框架。,source = demo
第2条数据：
id : 1,vector = [-0.0345064140856266, 0.05864095687866211, -0.012167001143097878, 0.03689992055296898, 0.0192620437592268],text = Milvus 是一个适合 AI 应用的向量数据库。,source = demo
第3条数据：
id : 2,vector = [-0.0401918925344944, -0.015303953550755978, 0.0043546287342906, -0.024416347965598106, 0.0164601169526577],text = RAG 的核心是先检索相关知识，再让大模型生成答案。,source = demo
第4条数据：
id : 3,vector = [-0.01210562139749527, 0.07837200909852982, -0.01580328308045864, 0.04114990681409836, 0.01612548530101776],text = Docker Desktop 可以方便地在本地运行 Milvus Standalone。,source = demo


In [30]:

res = client.get(
    collection_name=collection_name,
    ids=[0,1,2]
)

print(len(res))

for i in range(len(res)):
    print(f"第{i + 1}条数据：")
    print(f"id : {res[i]["id"]},vector = {res[i]["vector"][:5]},text = {res[i]["text"]},source = {res[i]["source"]}")
    # print(res[i])

3
第1条数据：
id : 0,vector = [-0.06995262950658798, -0.03449718654155731, -0.038969043642282486, 0.0352691151201725, -0.02789587341248989],text = LangChain 是一个用于构建 LLM 应用的开发框架。,source = demo
第2条数据：
id : 1,vector = [-0.0345064140856266, 0.05864095687866211, -0.012167001143097878, 0.03689992055296898, 0.0192620437592268],text = Milvus 是一个适合 AI 应用的向量数据库。,source = demo
第3条数据：
id : 2,vector = [-0.0401918925344944, -0.015303953550755978, 0.0043546287342906, -0.024416347965598106, 0.0164601169526577],text = RAG 的核心是先检索相关知识，再让大模型生成答案。,source = demo


In [31]:
# 相似度检索
query = "什么是向量数据库？"
query_vector = embed_model.embed_query(query)

In [32]:
results = client.search(
    collection_name=collection_name,
    data=[query_vector],
    limit=3,
    output_fields=["text","source","id"]
)

for res in results[0]:
    print(res)

{'id': 1, 'distance': 0.723461925983429, 'entity': {'id': 1, 'text': 'Milvus 是一个适合 AI 应用的向量数据库。', 'source': 'demo'}}
{'id': 3, 'distance': 0.4631018042564392, 'entity': {'id': 3, 'text': 'Docker Desktop 可以方便地在本地运行 Milvus Standalone。', 'source': 'demo'}}
{'id': 2, 'distance': 0.3883225619792938, 'entity': {'id': 2, 'text': 'RAG 的核心是先检索相关知识，再让大模型生成答案。', 'source': 'demo'}}
